![Node & Norm: Catholic institutional AI research](assets/book-banner.svg)

# 16. Prompt and output trace: prospective execution gate

**Control specification · 8 November 2026 · Development instrument, not an executed evaluation**

[Contents](00_START_HERE.ipynb) · [Governing protocol](02_Research_Protocol.ipynb) · [Reproduction](09_Reproduce_and_Inspect.ipynb) · [AI use](07_AI_Disclosure.ipynb) · [Owner gate #41](https://github.com/node-and-norm/control-evidence-corpus/issues/41)

**Question:** Can another researcher establish exactly what our team submitted to a model or connected tool, what the caller actually received, how failures/retries occurred, and what remains unavailable?

**Decision:** Make caller-visible prompt/output capture **mandatory before the next confirmatory study run**. The main research protocol 0.3.0 is still **unfrozen**, and no qualified reviewer or institutional owner has approved a new experiment.

The policy record is [_evidence/trace-policy/next-run-gate.json](_evidence/trace-policy/next-run-gate.json). Its proposed run manifest has **all approval switches false** and no runnable plan; those omissions are intentional.


## Why this matters to Catholic leaders and researchers

A claim that an AI assistant "refused an inappropriate request" must be traceable to an **exact prompt** and its **actual returned text**. If the assistant says to ask a priest for help, the trace establishes that the advice was written. It does **not** establish that the referral was delivered or acted upon.

Similarly, a reviewer may correct an interpretation of a Church source. The original answer, the proposed correction, the reviewer's decision and any subsequent action must remain **distinct records**. No retrospective editing of evidence or synthetic recreation of unavailable model output is permitted.

The capture tool makes this distinction inspectable; it does not verify theological correctness, establish legitimate canonical authority, or prove human-control effectiveness.


## How capture fits our six-stage workflow

| Research stage | Required trace |
|---|---|
| **01 Discover** | Exact search/query arguments, source IDs, returned metadata, rights/versions, and missing providers |
| **02 Screen** | Case inclusion/exclusion decisions, prompts used for screening, original evidence, AI use, reviewer exposure |
| **03 Review** | Human initial judgments, expertise, prompts shown, dissent and qualified interpretation recorded separately |
| **04 Reconstruct** | Frozen source packet, authority boundaries, actual opportunities to intervene, evidence missingness |
| **05 Evaluate** | **Pre-execution gate**, every submitted prompt/tool input, output/error, retry, reference record and frozen endpoint |
| **06 Report** | Reproducible trace manifest, full retained failures, exact source links, disclosures and limitations |

[See the visual research workflow](01_Study_and_Roadmap.ipynb). The recorder applies to future work; it does not turn historical conversations into raw transport logs.


## Minimum caller-visible capture fields

| Required field | How it is preserved | Limitation |
|---|---|---|
| Prompt text | Exact UTF-8 `prompt_text` and SHA-256, before call | Hidden system instructions are not observable |
| Request arguments | Caller-visible JSON `visible_args`, unchanged and hashed | Canonical JSON hashing is **not** proof of raw HTTP body bytes |
| Tool and provider | Tool name, kind, provider and case/attempt IDs | Internal provider routing may be hidden |
| Model and settings | Actual ID/parameters when visible; otherwise `null`, state and reason | Never guess missing model snapshots |
| Visible output | JSON result and unmodified returned answer text, hashed | May differ from internal provider processing |
| Failed calls | Error type and explicit privacy-driven omission; retries get new IDs | Exceptions can contain secrets, so raw text may require private custody |
| Attempt chronology | Start and terminal events with UTC client timestamps | Client timestamps are not provider latency |
| Integrity | Append-only JSONL with fsync and SHA-256 linked events | Hash chains detect later edits relative to a stored checkpoint; Git identity protects committed snapshots |
| Human review | Separate qualifications, initial ratings, AI exposure, approvals and dissent | Absence is recorded as pending, never assumed complete |

**Sensitive-data boundary:** Public GitHub is suitable only for synthetic or appropriately rights-cleared records. The basic credential-key check is a precaution, not a substitute for privacy screening. Any withheld sensitive output makes the public capture incomplete and blocks results promotion until secure, separately authorized custody is documented.


## Fail-closed pre-execution gate

Before a future model, agent or tool call may be invoked by the new runner, it must find a local `manifest.json` and matching frozen `plan.json` in that run's directory. The Python validator requires:

1. A named research-owner decision linked to a dated record.
2. Frozen protocol, case membership, expected labels' origin and source plan SHA-256, plus a fixed Git freeze commit.
3. Rights-cleared sources and approved public/private evidence handling.
4. Recorded privacy and ethics assessment, including any participant consent requirements.
5. A reviewer-independence, expertise and AI-exposure plan.
6. Predeclared outcomes, denominators, missingness, attempt limits and stopping/budget constraints.

Every gate remains pending in [issue #41](https://github.com/node-and-norm/control-evidence-corpus/issues/41). The checked-in proposed manifest intentionally **fails** preflight. A GitHub check that passes the source-contract tests does **not** authorize the next experiment.

The schema uses `run_kind: "confirmatory"` for real tests. A separate `offline_fixture` mode is available only for software unit tests with fake functions, explicitly labeled as nonresearch.


## Recorder interface for a future approved run

The actual Python module lives at [_tools/prompt_trace.py](_tools/prompt_trace.py). It is a synchronous wrapper: it saves the request **before** invoking the function and a success/error/incomplete terminal event **after** the function returns or fails.

The example below is **illustrative source code**. It is not a record of a model call and must not be executed against a real tool until the G0 approval and frozen run plan exist.

```python
from pathlib import Path
from catholic_trace import record_call  # import the checked-in module in your runner

run_dir = Path("catholic-ai-pilot/_evidence/trace-runs/NEW-FROZEN-RUN")
prompt = "Exact approved question as submitted"
args = {"prompt": prompt}

result = record_call(
    run_dir,
    call_id="NEW-RUN-case-001-attempt-1",
    case_id="case-001",
    attempt=1,
    kind="model",
    tool_name="approved.provider.chat",
    prompt_text=prompt,
    visible_args=args,
    provenance={
        "provider": "approved.provider",
        "model_id": None,
        "model_id_state": "not_exposed",
        "model_id_unavailable_reason": "The provider did not report it",
        "generation_parameters": None,
        "generation_parameters_state": "not_exposed",
        "parameters_unavailable_reason": "The provider did not report them",
        "transport_bytes_state": "not_captured",
    },
    invoke=lambda captured_args: provider.chat(**captured_args),
    response_text_from=lambda visible: visible["answer"],
)
```

The import line represents the eventual runner's checked-in import setup; it is not a public package called `catholic_trace`. Until an approved runner is implemented, researchers can load the module from `_tools/prompt_trace.py` using an explicit local path. **Never call a service through a separate untraced path and claim this module captured it.**


## Record lifecycle and offline verification

<text>Each call records a unique ID:</text>

```
approved plan + owner gate
            ↓
       START event
       exact prompt/args
       hash & fsync
            ↓
   tool/model invocation
          ↙    ↘
 SUCCESS / ERROR / INCOMPLETE
   output      explicit failure/absence
      ↓
  verify entire chain
      ↓
  release gate: no open or incomplete events
```

**Retries** are new attempts, not replacements. A failed attempt remains in sequence. A process crash after START leaves an open request and blocks the release check.

From a checked-out repository containing an **approved** run directory:

```bash
python catholic-ai-pilot/_tools/prompt_trace.py preflight catholic-ai-pilot/_evidence/trace-runs/NEW-FROZEN-RUN
python catholic-ai-pilot/_tools/prompt_trace.py verify catholic-ai-pilot/_evidence/trace-runs/NEW-FROZEN-RUN
python catholic-ai-pilot/_tools/prompt_trace.py promote-check catholic-ai-pilot/_evidence/trace-runs/NEW-FROZEN-RUN
```

The last command checks *capture completeness*, not scientific validity. Failing outputs remain retained for analysis. The offline tests simulate call success, error, retry, corrupt hash, missing answer and blocked preflight without making external requests.


## Historical capture: what is actually retained

| Existing work | Evidence retained | What must **not** be claimed |
|---|---|---|
| **PARISH-QA-001-R1** | Six exact caller-submitted prompts and complete visible answer/reference objects, with local pre/post timestamps | Not raw HTTP response bytes, hidden instructions, provider internal retrieval or verified theological accuracy |
| **CIR-001** | Frozen authored cases, deterministic engine, local run receipt and replay result evidence | Not a live model evaluation or independently validated human review |
| **AI-assisted source review** | Nineteen candidate claim-source assessments, source links and disclosure | Not the complete original prompt/output log of every assistant planning step |
| **Prior-work screening** | Seven-source registry, paper links, notebook synthesis and AI-use disclosure | Not an exhaustive peer-reviewed literature search or verbatim internal reasoning trace |

The older evidence remains exactly as it was recorded, with its missingness disclosed. This prospective contract never rewrites historical runs to simulate fuller capture.


## Failure criteria and protocol decisions

The system must reject or quarantine a prospective call when a plan has no recorded approval, has a changed SHA, attempts an unapproved tool, exceeds the frozen budget, omits a prompt, mutates an earlier call ID, produces an unserializable result or cannot retain the visible answer. Error records remain visible as failed attempts.

**What this cannot enforce by itself:** That a reviewer genuinely consented, a named approver had legitimate authority, source rights were legally sufficient, or the true provider internals were exposed. The runner can verify required fields and recorded decisions; qualified humans must verify their truth.

**Need owner review:** [Issue #41](https://github.com/node-and-norm/control-evidence-corpus/issues/41) remains the human decision point. The module and notebook are an auditable prospective contract, **not** a frozen experiment, new result or approval. Its limitations and AI contribution are listed in [the trace policy record](_evidence/trace-policy/next-run-gate.json).
